In [1]:
import io
import json
import logging as lg
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
import re
from sklearn.metrics.pairwise import paired_distances
from tqdm.auto import tqdm, trange
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import seaborn as sns
from collections import Counter
import igraph as ig
import pickle
import gzip
from sklearn.cluster import AgglomerativeClustering, HDBSCAN
from sklearn.neighbors import NearestNeighbors as NN


In [2]:
## import data
with gzip.open('pt_data.pkl.gz', 'rb') as fp:
    process_df, Trees = pickle.load(fp)

## revisiting -- process tree clustering via bag of features

We consider subtrees labelled with the root's process name.
We consider various ways to build bag of features (BoF)

* Concatenate the following:
  * process name
  * layer (distance from root)
  * counter (useful to consider multiple occurrences of the same process at the same layer)
* Bag of edges (using the process names)
* Bag of process name + out degree

Observations:

* First method clusters much better w.r.t. root processes
* Second and third methods have more badusers in baduser's NNs


In [3]:
## compute bag of features
def subtree_features(sg):
    root = np.where(np.array(sg.degree(mode='in'))==0)[0][0]
    V, L, _ = sg.bfs(root)
    F = []
    for i in range(len(L)-1):
        P = []
        for j in np.arange(L[i],L[i+1]):
            P.append(sg.vs[V[j]]['process'])
        C = Counter(P)
        keys = list(C.keys())
        vals = list(C.values())
        for j in range(len(keys)):
            for k in range(vals[j]):
                F.append(str(i)+'::'+keys[j]+'::'+str(k)) 
    return set(F)

## overlap coefficient (aka Szymkiewicz-Simpson)
def my_sim(a,b):
    return len(a.intersection(b))/min(len(a),len(b))

## utility function -- getting a subtree from Trees object
def get_subtree(Trees, tree_id, root_id):
    nodes, _ ,_ = Trees.subgraph(tree_id).bfs(root_id)
    g = Trees.subgraph(tree_id).subgraph(nodes)
    g["ly"] = g.layout_reingold_tilford() ## save a layout for plotting
    return g

## compute bag of features
def subtree_edge_features(sg):
    P = sg.vs['process']
    return set([ P[e.source]+'::'+P[e.target] for e in sg.es ])

def subtree_deg_features(sg):
    P = sg.vs['process']
    Dout = sg.degree(mode='out')
    F = [str(P[i])+'::'+str(Dout[i]) for i in np.arange(sg.vcount())]
    return set(F)

def subtree_plus_deg_features(sg):
    Dout = sg.degree(mode='out')
    root = np.where(np.array(sg.degree(mode='in'))==0)[0][0]
    V, L, _ = sg.bfs(root)
    F = []
    for i in range(len(L)-1):
        P = []
        for j in np.arange(L[i],L[i+1]):
            F.append(str(i)+'::'+str(sg.vs[V[j]]['process'])+'::'+str(Dout[V[j]]))
    ctr = Counter(F)
    return(set([k+'::'+str(v) for k,v in zip(ctr.keys(),ctr.values())]))


In [4]:
Features1 = []
Features2 = []
Features3 = []
Features4 = []
for t,r in zip(process_df.tree, process_df.root):
    sg = get_subtree(Trees, t, r)
    Features1.append(subtree_features(sg))
    Features2.append(subtree_edge_features(sg))
    Features3.append(subtree_deg_features(sg))
    Features4.append(subtree_plus_deg_features(sg))

with open('pt_features.pkl','wb') as fp:
    pickle.dump( (Features1, Features2, Features3, Features4), fp)


In [5]:
with open('pt_features.pkl','rb') as fp:
    F1, F2, F3, F4 = pickle.load(fp)


In [6]:
## consider the most common root process names
P = list(process_df['process'])
Counter(P).most_common(12)


[('bash.exe', 491),
 ('git.exe', 355),
 ('ngentask.exe', 304),
 ('cmd.exe', 241),
 ('taskhostw.exe', 151),
 ('compattelrunner.exe', 149),
 ('smss.exe', 94),
 ('winlogon.exe', 89),
 ('ssm-document-worker.exe', 84),
 ('explorer.exe', 44),
 ('userinit.exe', 43),
 ('ntoskrnl.exe', 41)]

In [7]:
## pick a feature set
Features = F1

In [8]:
top_process = 9
mc = set([x[0] for x in Counter(P).most_common(top_process)])
P_subset = [p for p in P if p in mc]
F_subset = [f for (p,f) in zip(P,Features) if p in mc]
B = list(process_df['has_bad'])
B_subset = [b for (p,b) in zip(P,B) if p in mc]

## compute similarities
l = len(F_subset)
S = np.zeros(shape=(l,l))
for i in range(len(F_subset)-1):
    for j in np.arange(i+1,len(F_subset)):
        s = my_sim(F_subset[i],F_subset[j])
        S[i,j] = S[j,i] = s

## cluster - set the correct number of clusters
model = AgglomerativeClustering(n_clusters=top_process, metric='precomputed', linkage='complete')
model.fit(1-S)

## list cluster compositions
for s in range(top_process):
    print( Counter([P_subset[i] for i in np.where(model.labels_==s)[0]]) , 'with bad:', Counter([B_subset[i] for i in np.where(model.labels_==s)[0]]))


Counter({'bash.exe': 482}) with bad: Counter({0: 482})
Counter({'smss.exe': 94}) with bad: Counter({0: 83, 1: 11})
Counter({'cmd.exe': 237}) with bad: Counter({0: 228, 1: 9})
Counter({'winlogon.exe': 89}) with bad: Counter({0: 81, 1: 8})
Counter({'taskhostw.exe': 151, 'cmd.exe': 3}) with bad: Counter({0: 154})
Counter({'ngentask.exe': 304, 'compattelrunner.exe': 149}) with bad: Counter({0: 453})
Counter({'bash.exe': 7, 'cmd.exe': 1}) with bad: Counter({0: 8})
Counter({'ssm-document-worker.exe': 84}) with bad: Counter({0: 84})
Counter({'git.exe': 355, 'bash.exe': 2}) with bad: Counter({0: 357})


In [9]:
model = HDBSCAN(metric='precomputed', min_cluster_size=15)
model.fit(1-S)
for s in np.arange(-1,max(model.labels_)+1):
    print(s, Counter([P_subset[i] for i in np.where(model.labels_==s)[0]]), 'with bad:', Counter([B_subset[i] for i in np.where(model.labels_==s)[0]]))


-1 Counter({'cmd.exe': 41, 'smss.exe': 11, 'bash.exe': 6}) with bad: Counter({0: 52, 1: 6})
0 Counter({'smss.exe': 83, 'cmd.exe': 12}) with bad: Counter({0: 87, 1: 8})
1 Counter({'git.exe': 355, 'bash.exe': 2}) with bad: Counter({0: 357})
2 Counter({'compattelrunner.exe': 149}) with bad: Counter({0: 149})
3 Counter({'ssm-document-worker.exe': 84, 'cmd.exe': 1}) with bad: Counter({0: 85})
4 Counter({'taskhostw.exe': 151}) with bad: Counter({0: 151})
5 Counter({'ngentask.exe': 304}) with bad: Counter({0: 304})
6 Counter({'winlogon.exe': 89}) with bad: Counter({0: 81, 1: 8})
7 Counter({'cmd.exe': 32, 'bash.exe': 12}) with bad: Counter({0: 43, 1: 1})
8 Counter({'bash.exe': 34, 'cmd.exe': 1}) with bad: Counter({0: 35})
9 Counter({'bash.exe': 306}) with bad: Counter({0: 306})
10 Counter({'bash.exe': 131}) with bad: Counter({0: 131})
11 Counter({'cmd.exe': 38}) with bad: Counter({0: 38})
12 Counter({'cmd.exe': 34}) with bad: Counter({0: 33, 1: 1})
13 Counter({'cmd.exe': 46}) with bad: Counter

In [10]:
## compute similarities
l = len(Features)
S = np.zeros(shape=(l,l))
for i in range(len(Features)-1):
    for j in np.arange(i+1,len(Features)):
        s = my_sim(Features[i],Features[j])
        S[i,j] = S[j,i] = s
    
## NN model
model = NN(n_neighbors=11, metric='precomputed')
model.fit(1-S)

## size of intersections - NNs and 
_, indices = model.kneighbors()
Bad = set(np.where(process_df.has_bad)[0])
bad_nn = [len(set(v).intersection(Bad)) for v in indices]

## average number of NN trees with baduser
_temp = pd.DataFrame(np.array([list(process_df.has_bad) , bad_nn]).T,columns=['bad','bad_nn'])
_temp.groupby(by='bad').mean()


,bad_nn
bad,
0,0.281421
1,2.648855


In [11]:
F4[0]

{'0::ssm-document-worker.exe::6::1',
 '1::powershell.exe::0::3',
 '1::wmic.exe::0::3'}

In [12]:
F3[0]

{'powershell.exe::0', 'ssm-document-worker.exe::6', 'wmic.exe::0'}

In [13]:
F2[0]

{'ssm-document-worker.exe::powershell.exe',
 'ssm-document-worker.exe::wmic.exe'}

In [14]:
F1[0]

{'0::ssm-document-worker.exe::0',
 '1::powershell.exe::0',
 '1::powershell.exe::1',
 '1::powershell.exe::2',
 '1::wmic.exe::0',
 '1::wmic.exe::1',
 '1::wmic.exe::2'}

### Quick EDA

Top non-baduser trees with several baduser NNs have the same shape, a sequence of three 'wsl.exe' processes.
    

In [15]:
_temp.sort_values(by='bad_nn', ascending=False).head(10)


,bad,bad_nn
1729,0,9
1710,0,9
1712,0,9
1747,1,9
2207,1,9
810,1,9
1730,1,8
2500,0,8
1684,1,8
1672,0,8


In [17]:
list(process_df.process_dict)[1729]

{0: Counter({'wsl.exe': 1}),
 1: Counter({'wsl.exe': 1}),
 2: Counter({'wsl.exe': 1})}